# Importação de Bibliotecas

In [ ]:
from google.colab import drive

import os
import gc
from datetime import datetime
from math import trunc

import numpy as np
import pandas as pd


from tqdm.notebook import tqdm



# Abertura do Arquivo

In [ ]:
# Acesso ao drive pessoal
drive.mount('/content/drive')

Mounted at /content/drive


# Definição do grupos e modelos

In [ ]:
grupos = ["intents", "permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]

# Definição das features a serem utilizadas

In [ ]:
for nome_grupo in tqdm(grupos):
  caminho_raiz = f"DIRETORIO_BASE/src_shap/{nome_grupo}/"

  caminho_mlp = os.path.join(caminho_raiz, "mlp_limiar5/")
  caminho_rf = os.path.join(caminho_raiz, "rf/")
  caminho_xgb = os.path.join(caminho_raiz, "xgb/")

  arquivo_features_importantes_mlp = os.path.join(caminho_mlp, f"{nome_grupo}__mlp__ranking_global_SHAP.csv")
  arquivo_features_importantes_rf = os.path.join(caminho_rf, f"{nome_grupo}__rf__ranking_global_SHAP.csv")
  arquivo_features_importantes_xgb = os.path.join(caminho_xgb, f"{nome_grupo}__xgb__ranking_global_SHAP.csv")

  # Lendo os 3 arquivos de ranking (um por modelo)
  df_mlp = pd.read_csv(arquivo_features_importantes_mlp)
  df_rf = pd.read_csv(arquivo_features_importantes_rf)
  df_xgb = pd.read_csv(arquivo_features_importantes_xgb)

  # Merge pelas features (inner join: mantem apenas features presentes nos 3)
  df_merged = df_mlp.merge(df_rf, on="feature", suffixes=("_mlp", "_rf"))
  df_merged = df_merged.merge(df_xgb, on="feature")
  df_merged = df_merged.rename(columns={"importance_mean_abs_shap": "importance_mean_abs_shap_xgb"})

  # Verificacao de seguranca: o merge inner pode descartar features nao-comuns
  if not (len(df_mlp) == len(df_rf) == len(df_xgb) == len(df_merged)):
      print(f"  ATENCAO [{nome_grupo}]: tamanhos divergem apos merge -> "
            f"mlp={len(df_mlp)}, rf={len(df_rf)}, xgb={len(df_xgb)}, merged={len(df_merged)}")

  # --- NORMALIZACAO POR MODELO (antes de mediar) ---
  # Cada modelo tem escala propria de valores SHAP. Para a media entre modelos
  # ser justa, normaliza-se cada coluna pela sua soma -> importancia RELATIVA
  # (cada modelo passa a somar 1). Assim o limiar de corte tem o mesmo
  # significado nos 3 modelos e nao sofre vies de escala.
  col_mlp = "importance_mean_abs_shap_mlp"
  col_rf = "importance_mean_abs_shap_rf"
  col_xgb = "importance_mean_abs_shap_xgb"

  for col in [col_mlp, col_rf, col_xgb]:
      soma = df_merged[col].sum()
      df_merged[col + "_norm"] = df_merged[col] / soma if soma > 0 else 0.0

  # Media das importancias JA NORMALIZADAS (esta e a media correta para corte)
  df_merged["importance_media"] = df_merged[
      [col_mlp + "_norm", col_rf + "_norm", col_xgb + "_norm"]
  ].mean(axis=1)

  # Ordena pela media normalizada
  df_merged = df_merged.sort_values(by="importance_media", ascending=False)
  df_merged.to_csv(os.path.join(caminho_raiz, "features_importancias_medias_geral.csv"), index=False)

  # Filtra apenas features com media de importancia > 0
  df_final = df_merged[df_merged["importance_media"] > 0]
  df_final = df_final.sort_values(by="importance_media", ascending=False)
  df_final.to_csv(os.path.join(caminho_raiz, "features_importancias_medias_nao_zero.csv"), index=False)
